# Day 2

Today, we will start using nf-core pipelines to find differentially abundant genes in our dataset. 
We are using data from the following paper: https://www.nature.com/articles/s41593-023-01350-3#Sec10

1. Please take some time to read through the paper and understand their approach, hypotheses and goals.

What was the objective of the study?

The objective was to look at causes for physical dependence and addiction disorders of opioid analgesics in pain therapeutics.
Hypothesis: Regulatory pathways differentially expressed in mice treated with oxycodon

What do the conditions mean?

oxy: mice treated with oxycodon  


sal: mice treated with saline solution

What do the genotypes mean?

SNI: This is the group, on which Spared nerve injury (SNI) surgery was performed to simulate persisting human neuropathic pain.

Sham: For this group, no SNI surgery was performed, leaving the mice with no persisting neuropathic pain. Nevertheless, this group had to undergo the same surgery procedure as for the SNI group.

Imagine you are the bioinformatician in the group who conducted this study. They hand you the raw files and ask you to analyze them.

What would you do?

Which groups would you compare to each other?

Please also mention which outcome you would expect to see from each comparison.

##### What I would do:
First of all, I would perform quality control (qc) to remove technical bias, biological noise and generally datapoints, which might distort further analysis

Next, I would perform normalisation (for example log, zscore, CPM/TPM).

Following this, I would perform differential abundance analysis for the given groups (Gene density, boxplots, heatmaps for highly variable genes).

##### Which groups I would compare to eachother:
First of, I would start to compare within the SNI and Sham genotypes for differences between the saline and the oxycodon group, to look at the influence of oxycodon on the groups.
Next, I would compare within the Oxy and Sal conditions, to look at the influence of the SNI surgery on the mice body and the oxycodon treatment.
Lastly, I would take a look at the SNI-Oxy vs Sham-Sal group to get a clear difference between mice with pain and oxycodon treatment, or the 'control/healthy' mice with no pain and oxycodon treatment.

##### Which outcome would I expect:
For the comparison within the SNI and Sal group, so comparing oxycodon vs. saline, I would expect to see difference in expression for the oxcodon treated micce.
For the SNI vs. Sham genotype within the Oxy and Sal group, I wouldn't expect much difference genewise because of the surgery. Maybe a higher expression in pain relevant genes.
Lastly, I would expect a clear and strong gene abundance difference between the SNI-Oxy and Sham-Sal groups.

Your group gave you a very suboptimal excel sheet (conditions_runs_oxy_project.xlsx) to get the information you need for each run they uploaded to the SRA.<br>
So, instead of directly diving into downloading the data and starting the analysis, you first need to sort the lazy table.<br>
Use Python and Pandas to get the table into a more sensible order.<br>
Then, perform some overview analysis and plot the results
1. How many samples do you have per condition?
2. How many samples do you have per genotype?
3. How often do you have each condition per genotype?

In [53]:
# load excel file into pandas
import pandas as pd

cond_file = pd.read_excel("./conditions_runs_oxy_project.xlsx")

# sort by Genotype SNI and then Oxy condition
cond_file = cond_file.sort_values(by=['Genotype: SNI','Condition: Oxy'])
print(f"sorted table:\n{cond_file}")

sorted table:
   Patient          Run RNA-seq  DNA-seq condition: Sal Condition: Oxy  \
3        ?  SRR23195508       x      NaN            NaN              x   
4        ?  SRR23195509       x      NaN            NaN              x   
11       ?  SRR23195516       x      NaN            NaN              x   
12       ?  SRR23195517       x      NaN            NaN              x   
0        ?  SRR23195505       x      NaN              x            NaN   
5        ?  SRR23195510       x      NaN              x            NaN   
8        ?  SRR23195513       x      NaN              x            NaN   
13       ?  SRR23195518       x      NaN              x            NaN   
1        ?  SRR23195506       x      NaN            NaN              x   
6        ?  SRR23195511       x      NaN            NaN              x   
9        ?  SRR23195514       x      NaN            NaN              x   
14       ?  SRR23195519       x      NaN            NaN              x   
2        ?  SRR23195507 

In [54]:

# count amount of samples per genotyp and condition
print(f"Amount of samples per genotype:\nSNI : {cond_file['Genotype: SNI'].value_counts().loc["x"]}\nSham: {cond_file['Genotype: Sham'].value_counts().loc["x"]}\n")
print(f"Amount of samples per condition:\nOxy: {cond_file['Condition: Oxy'].value_counts().loc["x"]}\nSal: {cond_file['condition: Sal'].value_counts().loc["x"]}\n")

print(f"Amount of condition per SNI:\nOxy:{cond_file.value_counts(["Genotype: SNI","Condition: Oxy"]).loc["x"].loc["x"]}\nSal:{cond_file.value_counts(["Genotype: SNI","condition: Sal"]).loc["x"].loc["x"]}\n")
print(f"Amount of condition per Sham:\nOxy:{cond_file.value_counts(["Genotype: Sham","Condition: Oxy"]).loc["x"].loc["x"]}\nSal:{cond_file.value_counts(["Genotype: Sham","condition: Sal"]).loc["x"].loc["x"]}")

Amount of samples per genotype:
SNI : 8
Sham: 8

Amount of samples per condition:
Oxy: 8
Sal: 8

Amount of condition per SNI:
Oxy:4
Sal:4

Amount of condition per Sham:
Oxy:4
Sal:4


They were so kind to also provide you with the information of the number of bases per run, so that you can know how much space the data will take on your Cluster.<br>
Add a new column to your fancy table with this information (base_counts.csv) and sort your dataframe according to this information and the condition.

Then select the 2 smallest runs from your dataset and download them from SRA (maybe an nf-core pipeline can help here?...)

In [ ]:
# read in base_counts csv file
base_counts = pd.read_csv("./base_counts.csv")

merged_file = cond_file.merge(
    how="left",
    right=base_counts,
    left_on="Run",  
    right_on="Run"  
) 
print(merged_file.sort_values(by="Bases", ascending=True).iloc[:2,:])
# two smallest SRA runs:
# SRR23195516
# SRR23195511 
merged_file.sort_values(by="Bases", ascending=True).iloc[:2,1].to_csv("./SRA_runs/SRA_runs.csv", index=False, header=False)

  Patient          Run RNA-seq  DNA-seq condition: Sal Condition: Oxy  \
2       ?  SRR23195516       x      NaN            NaN              x   
9       ?  SRR23195511       x      NaN            NaN              x   

  Genotype: SNI Genotype: Sham       Bases  
2             x            NaN  6203117700  
9           NaN              x  6456390900  


Download SRA run file:
Use nf-core pipeline (url:"https://nf-co.re/fetchngs/1.13.0/", date: 29.09.2026) fethngs to download SRAs:

!nextflow run nf-core/fetchngs -r 1.13.0 -profile podman --input ./SRA_runs/SRA_runs.csv --outdir ./SRA_runs

While your files are downloading, get back to the paper and explain how you would try to reproduce the analysis.<br>
When you are done with this shout, so we can discuss the different ideas.

First of all, if not already available, I would download all available datasets needed for reprodcution. For this, I would also take a look at the data and its format. So what data am I handling right now (scRNA-seq, bulk RNA-seq?).
After downloading all data, I would go to the Methods part in the paper and search for the analysis paragraph. Then I would try to replicate every step, if possible, to get as close as I can. This means replicating the whole workflow with qc, normalisation and analysis. If there are gaps, I would use widely used methods/tools for that part (nf-core pipelines if available to save time).
# TODO